# Proyecto Final de Minería de Datos
## Bank Marketing — Notebook de Creación, Ingeniería de Características y Preparación

**Integrantes:** Mariana Arbeláez, María Lucía Cadavid y Juan Manuel Arias  

**Fuente:** [UCI Machine Learning Repository — Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing)

### Alcance de este notebook
Este notebook retoma el conjunto de datos limpio de la Fase 1 (`bank_marketing_limpio_sin_balancear.csv`) y ejecuta:
1. Exclusión formal de la variable `duration` por motivos de fuga de información (*data leakage*).
2. Definición y aplicación de la ingeniería de características (codificación categórica y escalado numérico).
3. Partición del dataset en **Train (80%)** y **Test (20%)**.
4. Balanceo de clases avanzado combinando **Undersampling y Oversampling (SMOTE)** aplicado **únicamente sobre el conjunto de entrenamiento (80%)**.
5. Construcción y exportación del **Pipeline de preparación (`sklearn`)** para que sea consumido en los modelos de clasificación y ensambles.

In [1]:
%pip -q install imbalanced-learn scikit-learn pandas numpy

## 1. Importación de Librerías

In [2]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")

OUTPUT_DIR = Path("entregables_preparacion_bank_marketing")
DATA_PATH = OUTPUT_DIR / "bank_marketing_limpio_sin_balancear.csv"

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


## 2. Carga del Dataset Limpio
Cargamos el archivo `bank_marketing_limpio_sin_balancear.csv` generado en la etapa de preparación de datos.

In [4]:
from pathlib import Path

# Carga del archivo limpio generado en 01_Preparacion_Datos_Bank_Marketing
input_path = Path("/content/bank_marketing_limpio_sin_balancear.csv")

if not input_path.exists():
    # Fallback por si se ejecuta en otra ruta
    input_path = Path("bank_marketing_limpio_sin_balancear.csv")

df = pd.read_csv(input_path)
print("Dimensiones del dataset cargado:", df.shape)
display(df.head())

Dimensiones del dataset cargado: (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


## 3. Exclusión de la variable `duration` (Fuga de datos / Data Leakage)
Como se estableció en el análisis de negocio, la variable `duration` representa la duración del último contacto telefónico.

Dado que la intención del modelo es predecir si un cliente contratará un depósito **antes** de realizar la llamada, `duration` solo se conoce después de realizar la interacción. Su inclusión artificialmente elevaría la precisión del modelo pero lo inutilizaría en un escenario real de despliegue.

In [5]:
if "duration" in df.columns:
    df = df.drop(columns=["duration"])
    print("Variable 'duration' eliminada correctamente para prevenir fuga de datos.")
else:
    print("La variable 'duration' ya no se encuentra en el conjunto de datos.")

print(f"Nuevas dimensiones del dataset: {df.shape}")

Variable 'duration' eliminada correctamente para prevenir fuga de datos.
Nuevas dimensiones del dataset: (45211, 16)


In [6]:
# Separar características (X) y variable objetivo (y)
X = df.drop(columns=["y"])
y = df["y"].apply(lambda val: 1 if str(val).strip().lower() in ["yes"] else 0)

print("Dimensiones de X (predictores):", X.shape)
print("Distribución de la variable objetivo (y):\n", y.value_counts(normalize=True) * 100)

Dimensiones de X (predictores): (45211, 15)
Distribución de la variable objetivo (y):
 y
0    88.30152
1    11.69848
Name: proportion, dtype: float64


## 4. Identificación de tipos de variables para el Pipeline

In [7]:
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print("Variables numéricas:", numeric_features)
print("Variables categóricas:", categorical_features)

Variables numéricas: ['age', 'balance', 'day_of_week', 'campaign', 'pdays', 'previous']
Variables categóricas: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


## 5. Partición del dataset estratificado (80% Entrenamiento, 20% Prueba)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Dimensiones de X_test: {X_test.shape}")
print("\nDistribución en y_train (%):\n", y_train.value_counts(normalize=True) * 100)

Dimensiones de X_train: (36168, 15)
Dimensiones de X_test: (9043, 15)

Distribución en y_train (%):
 y
0    88.301814
1    11.698186
Name: proportion, dtype: float64


## 6. Construcción del Pipeline de Preparación (Scikit-Learn)

In [10]:
# Transformador para numéricas (Escalado)
numeric_transformer = StandardScaler()

# Transformador para categóricas (OneHotEncoder)
categorical_transformer = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# Column Transformer que une ambas transformaciones
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Pipeline de preprocesamiento configurado correctamente.")
preprocessor

Pipeline de preprocesamiento configurado correctamente.


ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['age', 'balance', 'day_of_week', 'campaign',
                                  'pdays', 'previous']),
                                ('cat',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['job', 'marital', 'education', 'default',
                                  'housing', 'loan', 'contact', 'month',
                                  'poutcome'])])

## 7. Balanceo Controlado con SMOTE (Train 80%)
Para evitar la pérdida de información de la clase mayoritaria, se descarta el undersampling. Se aplica únicamente un sobremuestreo controlado (**SMOTE**) sobre el conjunto de entrenamiento (80%) fijando `sampling_strategy=0.5`.

Esto incrementa la clase minoritaria (`1`) hasta representar exactamente el 50% del tamaño de la clase mayoritaria (`0`).

In [15]:
# 1. Ajustar y transformar los datos con el preprocesador
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# 2. Aplicar SMOTE prudente (Llevar la clase 1 al 50% de la clase 0)
smote = SMOTE(sampling_strategy=0.5, random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_transformed, y_train)

print("Dimensiones de entrenamiento antes del balanceo:", X_train_transformed.shape)
print("Dimensiones de entrenamiento después del SMOTE:", X_train_balanced.shape)
print("\nDistribución final de clases en y_train_balanced:\n", pd.Series(y_train_balanced).value_counts())
print("\nVerificación de proporciones (%):\n", pd.Series(y_train_balanced).value_counts(normalize=True) * 100)

Dimensiones de entrenamiento antes del balanceo: (36168, 50)
Dimensiones de entrenamiento después del SMOTE: (47905, 50)

Distribución final de clases en y_train_balanced:
 y
0    31937
1    15968
Name: count, dtype: int64

Verificación de proporciones (%):
 y
0    66.667362
1    33.332638
Name: proportion, dtype: float64
